In [ ]:
import numpy as np
import pygimli as pg
from pygimli.physics import ert
import matplotlib.pyplot as plt
import os
from datetime import datetime

# 1. Chargement et prétraitement des données
default_path = r"C:\Users\conde\Music\Datos Tomografía Eléctrica Santa Antonieta\dat - para Res2Dinv\Perfil 1.dat"

data = ert.load(default_path)
print(data)

# Calcul du facteur géométrique et de la résistivité apparente
data['k'] = ert.createGeometricFactors(data)
data['rho'] = data['u'] / data['i'] * data['k']

# Estimation des erreurs
data['err'] = pg.physics.ert.estimateError(data)

# Sauvegarde des données traitées
data.save("data.dat")

# 2. Création du gestionnaire ERT
manager = pg.physics.ert.ERTManager(
    data,
    sr=True,
    verbose=True,
    maxIter=4)

# 3. Inversion des données
inversion = manager.invert(
    lam=10,
    paraMaxCellSize=1)

# 4. Affichage du modèle inversé
ax, cb = manager.showResult(
    cMin=3,
    cMax=150,
    xlabel="x (m)",
    ylabel="z (m)",
    cMap='jet')

# ==========================================================
# SAUVEGARDE AUTOMATIQUE DE L'IMAGE
# ==========================================================

# Dossier de sortie
output_dir = r"C:\Users\conde\Documents\Resultats_ERT"

# Création du dossier si nécessaire
os.makedirs(output_dir, exist_ok=True)

# Nom du fichier avec date et heure
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

png_file = os.path.join(
    output_dir,
    f"Profil1_ERT_{timestamp}.png")

pdf_file = os.path.join(
    output_dir,
    f"Profil1_ERT_{timestamp}.pdf")

# Récupération de la figure
fig = ax.figure

# Sauvegarde
fig.savefig(png_file, dpi=300, bbox_inches='tight')
fig.savefig(pdf_file, bbox_inches='tight')

print("\nImage PNG sauvegardée :")
print(png_file)

print("\nImage PDF sauvegardée :")
print(pdf_file)

# ==========================================================
# INDICATEURS DE QUALITE
# ==========================================================

print("\nRMS Error :", manager.inv.relrms(), "%")
print("Chi² value :", manager.inv.chi2())

plt.show()